# Bias Analysis — Credit Approval Decisions

This notebook evaluates potential bias in historical credit approval outcomes, focusing on **gender-based disparate impact (DI)** and **proxy discrimination**.

## 1. Data & Setup

We run the analysis on the cleaned / analysis-ready dataset produced by the data-quality pipeline.  
We report sample size and confirm the variables required for bias analysis.

In [20]:
import sys
from pathlib import Path
import importlib

sys.path.append(str(Path("..").resolve()))

import src.data_clean_notebook as preproc
importlib.reload(preproc)

df = preproc.run_data_quality_pipeline("../data/raw_credit_applications.json")

Final dataset shape: (502, 27)


In [21]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())

shape: (502, 27)
columns: ['applicant_gender', 'applicant_zip_code', 'fin_annual_income', 'fin_credit_history_months', 'fin_debt_to_income', 'fin_savings_balance', 'decision_loan_approved', 'spend_shopping', 'spend_rent', 'spend_alcohol', 'spend_txn_count', 'spend_total', 'spend_mean', 'spend_max', 'spend_unique_cats', 'spend_dining', 'spend_healthcare', 'spend_fitness', 'spend_entertainment', 'spend_insurance', 'spend_travel', 'spend_transportation', 'spend_utilities', 'spend_groceries', 'spend_education', 'spend_adult_entertainment', 'spend_gambling']


In [22]:
# Minimal sanity checks
print("\nRaw gender counts:")
print(df["applicant_gender"].value_counts(dropna=False))

print("\nRaw label counts:")
print(df["decision_loan_approved"].value_counts(dropna=False))


Raw gender counts:
applicant_gender
Male      195
Female    193
F          58
M          53
NaN         3
Name: count, dtype: int64

Raw label counts:
decision_loan_approved
True     292
False    210
Name: count, dtype: int64


## 2. Definitions

- **Outcome:** loan approval label `y` (1 = approved, 0 = rejected) derived from `decision_loan_approved`.
- **Protected attribute:** gender derived from `applicant_gender` and normalized into `gender_norm` (male/female).
- **Selection rate:** group approval rate, used for DI computation.

Fairness metrics are computed only on rows where both the protected attribute and outcome are defined.

## 3. Missingness & Validity Checks (Protected Attribute)

Before computing fairness metrics, we audit missingness in the protected attribute.  
If missingness correlates with outcomes, excluding missing rows may distort fairness estimates.

We report:
- missing rate for `applicant_gender`
- outcome distribution and approval rate for missing vs non-missing rows

In [24]:
# Missingness audit on raw applicant_gender

raw_gender = df["applicant_gender"]
miss_raw_gender = raw_gender.isna() | (raw_gender.astype(str).str.strip() == "")

print(f"Missing applicant_gender rate: {miss_raw_gender.mean():.4%}")
print("Counts (missing vs non-missing):")
print(miss_raw_gender.value_counts())

# If y exists, compare approval rate
df["y"] = df["decision_loan_approved"].astype(int)
print("\nApproval rate when raw gender is missing:", df.loc[miss_raw_gender, "y"].mean() if miss_raw_gender.any() else "N/A")
print("Approval rate when raw gender is NOT missing:", df.loc[~miss_raw_gender, "y"].mean())

Missing applicant_gender rate: 0.5976%
Counts (missing vs non-missing):
applicant_gender
False    499
True       3
Name: count, dtype: int64

Approval rate when raw gender is missing: 0.6666666666666666
Approval rate when raw gender is NOT missing: 0.5811623246492986


**Missingness audit**

DI is computed on records with non-missing protected attribute and outcome, because DI is undefined otherwise. The missing rate for gender_norm is 0.60% (3/502), which is minimal. The missing subset is too small to draw reliable conclusions about systematic missingness; therefore the DI findings are unlikely to be driven by missing-data handling.


## 4. Group Descriptives (Gender)

We report:
- group sample sizes
- selection/approval rates by gender

These descriptives provide context for DI and help interpret whether disparities are driven by base-rate differences.

We first compute approval/selection rates by group and then compute Disparate Impact (DI). DI is computed on rows with non-missing protected attribute and outcome, because DI is undefined otherwise. 

In [25]:
# Create gender_norm WITHOUT hiding missing values

g = df["applicant_gender"].astype("string").str.strip().str.lower()

gender_map = {
    "m": "male", "male": "male", "man": "male",
    "f": "female", "female": "female", "woman": "female",
}

df["gender_norm"] = g.map(gender_map)

# IMPORTANT: keep missing as <NA> (do NOT fillna here)
# df["gender_norm"] remains NA if raw gender was missing or unrecognized

# (optional) show what values are not mapped
print("gender_norm value counts (including NA):")
print(df["gender_norm"].value_counts(dropna=False))

gender_norm value counts (including NA):
gender_norm
female    251
male      248
NaN         3
Name: count, dtype: int64


## 5. Disparate Impact (DI) by Gender







### 5.1 Method
DI is computed as:

\[
DI = \frac{\text{selection rate of unprivileged group}}{\text{selection rate of privileged group}}
\]

We use the common convention:
- privileged = **male**
- unprivileged = **female**

We compute DI only on rows where both `gender_norm` and `y` are non-missing (DI is undefined otherwise).

In [28]:
# Prepare DI dataset (DI is undefined when gender is missing)

tmp = df[["gender_norm", "y"]].dropna().copy()
tmp["y"] = tmp["y"].astype(int)

print("DI dataset size:", tmp.shape[0])
print(tmp["gender_norm"].value_counts())

group_sizes = tmp["gender_norm"].value_counts()
approval_rates = tmp.groupby("gender_norm")["y"].mean().sort_values(ascending=False)

print("Group sizes:\n", group_sizes)
print("\nApproval rates:\n", approval_rates)

DI dataset size: 499
gender_norm
female    251
male      248
Name: count, dtype: int64
Group sizes:
 gender_norm
female    251
male      248
Name: count, dtype: int64

Approval rates:
 gender_norm
male      0.657258
female    0.505976
Name: y, dtype: float64


### 5.2 Computation (selection rates + DI)
We compute group sizes and selection rates, then calculate DI.

In [27]:
priv_group = "male"
unpriv_group = "female"

if priv_group in approval_rates.index and unpriv_group in approval_rates.index:
    di = float(approval_rates.loc[unpriv_group] / approval_rates.loc[priv_group])
    print(f"DI (female/male) = {di:.4f}")
    print(f"80% rule flag (DI < 0.8): {di < 0.8}")
else:
    print("Cannot compute DI: missing male or female group in the DI dataset.")
    print("Available groups:", approval_rates.index.tolist())

DI (female/male) = 0.7698
80% rule flag (DI < 0.8): True


### 5.3 Interpretation (Four-Fifths Rule)
We interpret DI using the four-fifths rule:
- **DI < 0.80** triggers a screening flag for potential adverse impact.
DI is descriptive and does not establish causality; therefore we complement it with proxy checks.

Conclusion — Gender DI (Disparate Impact)

Missingness check: applicant_gender is missing for 3/502 (0.60%) records. The approval rate for the missing-gender rows is 0.667 (2/3) vs 0.581 for non-missing rows. Because the missing group is extremely small, excluding these rows is unlikely to materially change the DI estimate, but we report it transparently.

Group selection rates: On the DI computation subset (n=499 with non-missing gender), the approval/selection rate is 0.657 for male (n=248) and 0.506 for female (n=251).

Disparate Impact (DI): Using the common convention DI = female_rate / male_rate, we obtain DI = 0.7698. This falls below the 0.80 four-fifths threshold, so the 80% rule is triggered, indicating potential adverse impact against the female group in the observed decision outcomes.

Note on interpretation: DI is a screening metric, not a causal proof of discrimination. Next steps include investigating possible proxy drivers (e.g., zip code or financial variables correlated with gender) and checking whether the disparity persists after controlling for relevant risk factors.

## 6. Statistical Association (Gender vs Approval)

To complement DI, we test whether approval outcomes are associated with gender using:
- **Chi-square test of independence** on the contingency table (gender × approval)
- **Cramer’s V** as an effect size (strength of association)

We test whether approval decisions are independent of gender.

H0: gender and approval are independent.

If p < 0.05, we reject H0 and conclude the difference is statistically significant.
This helps distinguish “numerical disparity” from “statistically supported association”.

### Chi-square test of independence

In [30]:
import pandas as pd
from scipy.stats import chi2_contingency

ct = pd.crosstab(tmp["gender_norm"], tmp["y"])
print("Contingency table (gender x approval):\n", ct)

chi2, p, dof, expected = chi2_contingency(ct)

print(f"\nChi-square statistic: {chi2:.4f}")
print(f"p-value: {p:.6f}")
print(f"Degrees of freedom: {dof}")

expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
print("\nExpected counts under H0 (independence):\n", expected_df.round(2))

Contingency table (gender x approval):
 y              0    1
gender_norm          
female       124  127
male          85  163

Chi-square statistic: 11.1156
p-value: 0.000856
Degrees of freedom: 1

Expected counts under H0 (independence):
 y                 0       1
gender_norm                
female       105.13  145.87
male         103.87  144.13


**Chi-Square conclusion**

The chi-square test indicates that loan approval decisions are not independent of gender (χ²(1)=11.12, p=0.000856). Therefore, we reject H0 and conclude that the difference in approval outcomes across gender groups is statistically significant. The observed pattern is consistent with the fairness metrics above: the approval rate is higher for Male than for Female, supporting the presence of disparate outcomes that warrant further investigation.

### Cramer's V test
We report Cramer’s V to quantify the strength of association between gender and approval.

In [31]:
import numpy as np

n = ct.to_numpy().sum()
r, k = ct.shape
cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))
print(f"Cramer's V: {cramers_v:.4f}")

Cramer's V: 0.1493


**Cramer's V Conclusion**

The effect size (Cramer’s V = 0.1493) suggests the gender–approval relationship is weak-to-moderate. This supports monitoring and audit actions: even modest associations can be problematic in decision systems, especially when DI violates the four-fifths rule.

## 7. Robustness / Sensitivity Checks

We run small checks to ensure the DI result is not an artifact of:
- gender normalization rules
- dropping a very small number of missing protected-attribute rows
- inclusion/exclusion of non-binary/unknown gender values

In [36]:
# Robustness check: DI computed on non-missing gender (baseline)
baseline_n = tmp.shape[0]
print("Baseline DI dataset size:", baseline_n)

# If there are very few missing gender rows, report it explicitly
missing_gender_n = df["gender_norm"].isna().sum()
print("Rows with missing/unmapped gender_norm:", int(missing_gender_n))

Baseline DI dataset size: 499
Rows with missing/unmapped gender_norm: 3


## 8. Proxy Discrimination Analysis

Even if gender is not explicitly used in decisions, non-protected variables may act as proxies.  
We structure proxy evidence as a chain:


### 8.1 Candidate proxies (hypotheses)
We consider variables such as ZIP code, financial attributes, and spending behavior as potential proxies.

### 8.2 Association with gender
We test whether candidate variables show meaningful differences across gender groups.

In [33]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# 8.2 Association with gender: ZIP code vs gender
proxy_df = df[["applicant_zip_code", "gender_norm"]].dropna().copy()
proxy_df["applicant_zip_code"] = proxy_df["applicant_zip_code"].astype(str)

ct_zip_gender = pd.crosstab(proxy_df["applicant_zip_code"], proxy_df["gender_norm"])
print("Contingency table shape (zip x gender):", ct_zip_gender.shape)

chi2_zg, p_zg, dof_zg, exp_zg = chi2_contingency(ct_zip_gender)
n_zg = ct_zip_gender.to_numpy().sum()
r_zg, k_zg = ct_zip_gender.shape
cramers_v_zg = np.sqrt(chi2_zg / (n_zg * (min(r_zg - 1, k_zg - 1))))

print(f"Chi-square (zip vs gender): {chi2_zg:.4f}")
print(f"p-value: {p_zg:.6f}")
print("Cramer's V: {cramers_v_zg:.4f}")

Contingency table shape (zip x gender): (195, 2)
Chi-square (zip vs gender): 393.7343
p-value: 0.000000
Cramer's V: {cramers_v_zg:.4f}


### 8.3 Link to approval outcomes
We test whether the same variables are also associated with approval decisions.

In [35]:
# 8.3 Link to approval outcomes: ZIP code vs approval
proxy_out_df = df[["applicant_zip_code", "y"]].dropna().copy()
proxy_out_df["applicant_zip_code"] = proxy_out_df["applicant_zip_code"].astype(str)
proxy_out_df["y"] = proxy_out_df["y"].astype(int)

ct_zip_y = pd.crosstab(proxy_out_df["applicant_zip_code"], proxy_out_df["y"])
chi2_zy, p_zy, dof_zy, exp_zy = chi2_contingency(ct_zip_y)

n_zy = ct_zip_y.to_numpy().sum()
r_zy, k_zy = ct_zip_y.shape
cramers_v_zy = np.sqrt(chi2_zy / (n_zy * (min(r_zy - 1, k_zy - 1))))

print(f"Chi-square (zip vs approval): {chi2_zy:.4f}")
print(f"p-value: {p_zy:.6f}")
print("Cramer's V: {cramers_v_zy:.4f}")

# show ZIP-level approval rates (with minimum support to avoid noise)
zip_stats = (
    proxy_out_df.groupby("applicant_zip_code")["y"]
    .agg(n="count", approval_rate="mean")
    .sort_values(["approval_rate", "n"], ascending=[False, False])
)

display(zip_stats.head(10))
display(zip_stats.tail(10))

Chi-square (zip vs approval): 181.0501
p-value: 0.738462
Cramer's V: {cramers_v_zy:.4f}


,n,approval_rate
applicant_zip_code,,
10004,6,1.0
10012,5,1.0
10092,5,1.0
10001,4,1.0
10040,4,1.0
10077,4,1.0
10005,3,1.0
10058,3,1.0
10071,3,1.0


,n,approval_rate
applicant_zip_code,,
90216,1,0.0
90220,1,0.0
90240,1,0.0
90241,1,0.0
90242,1,0.0
90249,1,0.0
90257,1,0.0
90259,1,0.0
90264,1,0.0


## 8.4 Proxy risk summary
A variable linked to both gender and approval outcomes is flagged as a proxy discrimination risk.

需修改

We evaluate ZIP code as a potential proxy feature.  
If ZIP code is statistically associated with **gender** and also with **approval outcomes**, it may act as a proxy pathway that indirectly encodes gender information in decisions.

Next, we prioritize investigating whether other features (e.g., income, debt-to-income, spending patterns) exhibit a similar “gender association + outcome association” pattern.

## 9. Limitations

- DI and statistical tests are descriptive and do not prove causality or intent.
- Dataset size and simulated artifacts may affect estimates.
- Unobserved confounders can explain some observed disparities.

## 10. Recommendations 

We propose actionable controls such as:
- monitoring DI over time with alerts (e.g., DI < 0.8)
- reviewing or justifying strong proxy variables (e.g., ZIP code)
- documentation of feature usage and decision rationale
- human oversight and audit trails for high-risk decisions

## 11. Key Findings Summary (for README / Video)

We summarize:
- group sizes and selection rates
- DI value and whether the 80% rule is triggered
- strongest proxy risk signals identified